# 01 — Annotation

Generate a standardized transcript annotation table from the reference annotation for downstream Ribo-seq analyses.

## Purpose

This notebook extracts transcript-level annotations from the reference genome annotation.

The resulting table serves as the backbone for all downstream analyses, including RNA quantification, Ribo-seq quantification, translation efficiency calculation, alignment summaries, and footprint analyses.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gzip
from collections import Counter

In [5]:
# ============================================
# Configuration
# ============================================

# Workflow repository
REPO = Path.home() / "Code" / "processing" / "riboseq"

# Reference resources
REFERENCE = REPO / "refs" / "built" / "arabidopsis_tair10_rsemclean"

# Project data
PROJECT = Path("/mnt/d/Ibnu/riboseq/AT")

# nf-core results
INPUT = PROJECT / "nfcore" / "10pct"

# Generated tables
TABLES = PROJECT / "TABLES"

# Optional figures
FIGURES = PROJECT / "FIGURES"

TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

In [9]:
# ============================================
# Input inspection
# ============================================

print(f"Repository       : {REPO}")
print(f"Reference        : {REFERENCE}")
print(f"Project directory: {PROJECT}")
print(f"Input directory  : {INPUT}")
print(f"Tables directory : {TABLES}")
print(f"Figures directory: {FIGURES}")

print()
print("Reference exists:", REFERENCE.exists())
print("Input exists    :", INPUT.exists())
print("Tables exists   :", TABLES.exists())
print("Figures exists  :", FIGURES.exists())

Repository       : /home/ha-ibnu/Code/processing/riboseq
Reference        : /home/ha-ibnu/Code/processing/riboseq/refs/built/arabidopsis_tair10_rsemclean
Project directory: /mnt/d/Ibnu/riboseq/AT
Input directory  : /mnt/d/Ibnu/riboseq/AT/nfcore/10pct
Tables directory : /mnt/d/Ibnu/riboseq/AT/TABLES
Figures directory: /mnt/d/Ibnu/riboseq/AT/FIGURES

Reference exists: True
Input exists    : True
Tables exists   : True
Figures exists  : True


## Annotation table design

The target output of this notebook is:

| Column | Description |
|--------|-------------|
| `Transcript_ID` | Transcript identifier |
| `Gene_ID` | Gene identifier |
| `Chr` | Chromosome or contig |
| `Strand` | Genomic strand |
| `Transcript_start` | Transcript start coordinate |
| `Transcript_end` | Transcript end coordinate |
| `Transcript_length` | Transcript length |
| `CDS_length` | Total CDS length |
| `UTR5_length` | Total 5′UTR length |
| `UTR3_length` | Total 3′UTR length |

In [11]:
# ============================================
# Load reference annotation
# ============================================
GTF = REFERENCE / "at.rsem.gtf"

print("Reference annotation")
print("--------------------")
print(GTF)
print("Exists:", GTF.exists())

Reference annotation
--------------------
/home/ha-ibnu/Code/processing/riboseq/refs/built/arabidopsis_tair10_rsemclean/at.rsem.gtf
Exists: True


In [12]:
with open(GTF) as f:
    for _ in range(20):
        print(f.readline().rstrip())

Chr1	Araport11	transcript	3631	5899	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010"
Chr1	Araport11	exon	3631	3913	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	3996	4276	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	4486	4605	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	4706	5095	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	5174	5326	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	5439	5899	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	3760	3913	.	+	0	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	3996	4276	.	+	2	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	4486	4605	.	+	0	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	4706	5095	.	+	0	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	5174	5326	.	+	0	transcript_id "AT1

In [13]:
# ============================================
# Inspect feature types
# ============================================

feature_counts = {}

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue
        feature = fields[2]
        feature_counts[feature] = feature_counts.get(feature, 0) + 1

pd.Series(feature_counts).sort_values(ascending=False)

exon          322737
CDS           286355
transcript     59478
dtype: int64

In [15]:

feature_counts = Counter()

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        if len(fields) != 9:
            continue

        feature_counts[fields[2]] += 1

pd.DataFrame(
    feature_counts.items(),
    columns=["Feature", "Count"]
).sort_values("Count", ascending=False).reset_index(drop=True)

,Feature,Count
0,exon,322737
1,CDS,286355
2,transcript,59478


In [16]:
# ============================================
# Inspect annotation attributes
# ============================================

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        print(fields[8])

        break

transcript_id "AT1G01010.1"; gene_id "AT1G01010"


In [17]:
# ============================================
# Attribute parser
# ============================================

def parse_gtf_attributes(attr: str) -> dict:
    """
    Parse a GTF attribute string.

    Example:
    transcript_id "AT1G01010.1"; gene_id "AT1G01010";
    """
    result = {}

    for item in attr.strip().split(";"):
        item = item.strip()

        if not item:
            continue

        key, value = item.split(" ", 1)
        result[key] = value.strip().strip('"')

    return result


# Test parser
example_attr = 'transcript_id "AT1G01010.1"; gene_id "AT1G01010";'
parse_gtf_attributes(example_attr)

{'transcript_id': 'AT1G01010.1', 'gene_id': 'AT1G01010'}

In [18]:
# ============================================
# Parse GTF records
# ============================================

records = []

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        if len(fields) != 9:
            continue

        chrom, source, feature, start, end, score, strand, frame, attr = fields
        attrs = parse_gtf_attributes(attr)

        records.append({
            "Chr": chrom,
            "Source": source,
            "Feature": feature,
            "Start": int(start),
            "End": int(end),
            "Strand": strand,
            "Transcript_ID": attrs.get("transcript_id"),
            "Gene_ID": attrs.get("gene_id"),
        })

gtf_df = pd.DataFrame(records)

gtf_df.head()

,Chr,Source,Feature,Start,End,Strand,Transcript_ID,Gene_ID
0,Chr1,Araport11,transcript,3631,5899,+,AT1G01010.1,AT1G01010
1,Chr1,Araport11,exon,3631,3913,+,AT1G01010.1,AT1G01010
2,Chr1,Araport11,exon,3996,4276,+,AT1G01010.1,AT1G01010
3,Chr1,Araport11,exon,4486,4605,+,AT1G01010.1,AT1G01010
4,Chr1,Araport11,exon,4706,5095,+,AT1G01010.1,AT1G01010


In [19]:
gtf_df.shape

(668570, 8)

In [20]:
# ============================================
# Validate parsed records
# ============================================

gtf_df["Feature"].value_counts()

Feature
exon          322737
CDS           286355
transcript     59478
Name: count, dtype: int64

In [21]:
# ============================================
# Check missing identifiers
# ============================================

gtf_df[["Transcript_ID", "Gene_ID"]].isna().sum()

Transcript_ID    0
Gene_ID          0
dtype: int64

In [22]:
# ============================================
# Transcript backbone
# ============================================

transcripts = (
    gtf_df[gtf_df["Feature"] == "transcript"]
    .copy()
    .rename(columns={
        "Start": "Transcript_start",
        "End": "Transcript_end",
    })
    [[
        "Transcript_ID",
        "Gene_ID",
        "Chr",
        "Strand",
        "Transcript_start",
        "Transcript_end",
    ]]
)

transcripts["Genomic_span_length"] = (
    transcripts["Transcript_end"] - transcripts["Transcript_start"] + 1
)

transcripts.head()

,Transcript_ID,Gene_ID,Chr,Strand,Transcript_start,Transcript_end,Genomic_span_length
0,AT1G01010.1,AT1G01010,Chr1,+,3631,5899,2269
13,AT1G01020.2,AT1G01020,Chr1,-,6788,8737,1950
29,AT1G01020.6,AT1G01020,Chr1,-,6788,8737,1950
40,AT1G01020.1,AT1G01020,Chr1,-,6788,9130,2343
59,AT1G01020.3,AT1G01020,Chr1,-,6788,9130,2343


In [23]:
transcripts.shape

(59478, 7)

In [24]:
# ============================================
# Exon-derived transcript length
# ============================================

exons = gtf_df[gtf_df["Feature"] == "exon"].copy()

exons["Exon_length"] = exons["End"] - exons["Start"] + 1

exon_lengths = (
    exons
    .groupby("Transcript_ID", as_index=False)["Exon_length"]
    .sum()
    .rename(columns={"Exon_length": "Transcript_length"})
)

exon_lengths.head()

,Transcript_ID,Transcript_length
0,AT1G01010.1,1688
1,AT1G01020.1,1329
2,AT1G01020.2,1087
3,AT1G01020.3,1420
4,AT1G01020.4,1397


In [25]:
exon_lengths.shape

(59051, 2)

In [26]:
# ============================================
# Check transcripts without exon records
# ============================================

missing_exon = transcripts.loc[
    ~transcripts["Transcript_ID"].isin(exon_lengths["Transcript_ID"])
]

missing_exon.shape

(427, 7)

In [27]:
missing_exon.head(10)

,Transcript_ID,Gene_ID,Chr,Strand,Transcript_start,Transcript_end,Genomic_span_length
207,ath-miR838,ath-miR838,Chr1,+,28635,28655,21
658,ath-miR165a-3p,ath-miR165a-3p,Chr1,-,78932,78952,21
659,ath-miR165a-5p,ath-miR165a-5p,Chr1,-,79010,79030,21
1759,ath-miR2112-3p,ath-miR2112-3p,Chr1,-,234017,234037,21
1760,ath-miR2112-5p,ath-miR2112-5p,Chr1,-,234126,234146,21
14215,ath-miR5640,ath-miR5640,Chr1,-,1653539,1653559,21
14743,ath-miR5656,ath-miR5656,Chr1,+,1727351,1727371,21
18525,ath-miR847,ath-miR847,Chr1,+,2165675,2165695,21
33771,ath-miR171b-3p,ath-miR171b-3p,Chr1,-,3961367,3961387,21
33772,ath-miR171b-5p,ath-miR171b-5p,Chr1,-,3961430,3961450,21


In [28]:
# ============================================
# CDS length
# ============================================

cds = gtf_df[gtf_df["Feature"] == "CDS"].copy()

cds["CDS_segment_length"] = cds["End"] - cds["Start"] + 1

cds_lengths = (
    cds
    .groupby("Transcript_ID", as_index=False)["CDS_segment_length"]
    .sum()
    .rename(columns={"CDS_segment_length": "CDS_length"})
)

cds_lengths.head()

,Transcript_ID,CDS_length
0,AT1G01010.1,1290
1,AT1G01020.1,738
2,AT1G01020.2,576
3,AT1G01020.3,711
4,AT1G01020.4,711


In [29]:
cds_lengths.shape

(48359, 2)

In [30]:
# ============================================
# Merge annotation information
# ============================================

annotation = (
    transcripts
    .merge(exon_lengths, on="Transcript_ID", how="left")
    .merge(cds_lengths, on="Transcript_ID", how="left")
)

annotation.head()

,Transcript_ID,Gene_ID,Chr,Strand,Transcript_start,Transcript_end,Genomic_span_length,Transcript_length,CDS_length
0,AT1G01010.1,AT1G01010,Chr1,+,3631,5899,2269,1688.0,1290.0
1,AT1G01020.2,AT1G01020,Chr1,-,6788,8737,1950,1087.0,576.0
2,AT1G01020.6,AT1G01020,Chr1,-,6788,8737,1950,944.0,315.0
3,AT1G01020.1,AT1G01020,Chr1,-,6788,9130,2343,1329.0,738.0
4,AT1G01020.3,AT1G01020,Chr1,-,6788,9130,2343,1420.0,711.0


In [31]:
# ============================================
# Calculate UTR length
# ============================================

annotation["Transcript_length"] = annotation["Transcript_length"].fillna(0)
annotation["CDS_length"] = annotation["CDS_length"].fillna(0)

annotation["UTR_length"] = (
    annotation["Transcript_length"] -
    annotation["CDS_length"]
)

annotation.head()

,Transcript_ID,Gene_ID,Chr,Strand,Transcript_start,Transcript_end,Genomic_span_length,Transcript_length,CDS_length,UTR_length
0,AT1G01010.1,AT1G01010,Chr1,+,3631,5899,2269,1688.0,1290.0,398.0
1,AT1G01020.2,AT1G01020,Chr1,-,6788,8737,1950,1087.0,576.0,511.0
2,AT1G01020.6,AT1G01020,Chr1,-,6788,8737,1950,944.0,315.0,629.0
3,AT1G01020.1,AT1G01020,Chr1,-,6788,9130,2343,1329.0,738.0,591.0
4,AT1G01020.3,AT1G01020,Chr1,-,6788,9130,2343,1420.0,711.0,709.0


In [32]:
# ============================================
# Build exon and CDS segment dictionaries
# ============================================

exon_segments = (
    exons
    .sort_values(["Transcript_ID", "Start", "End"])
    .groupby("Transcript_ID")[["Start", "End"]]
    .apply(lambda x: list(map(tuple, x.to_numpy())))
    .to_dict()
)

cds_segments = (
    cds
    .sort_values(["Transcript_ID", "Start", "End"])
    .groupby("Transcript_ID")[["Start", "End"]]
    .apply(lambda x: list(map(tuple, x.to_numpy())))
    .to_dict()
)

print("Transcript with exons:", len(exon_segments))
print("Transcript with CDS  :", len(cds_segments))

Transcript with exons: 59051
Transcript with CDS  : 48359
